# SAM3 API Server — VitroVision
FastAPI + SAM3 + ngrok bridge สำหรับ Android app

**Runtime:** ต้องเลือก GPU (Runtime → Change runtime type → T4 GPU)

In [ ]:
!pip install -q transformers torch torchvision opencv-python pillow numpy fastapi uvicorn nest-asyncio pyngrok requests

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch
import numpy as np
from PIL import Image
import cv2
from transformers import Sam3Processor, Sam3Model
from io import BytesIO
import base64

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

print("กำลังโหลด SAM3...")
model = Sam3Model.from_pretrained("facebook/sam3").to(device)
processor = Sam3Processor.from_pretrained("facebook/sam3")
print("โหลดเสร็จ!")

In [ ]:
def segment_image(image_pil, prompts):
    if isinstance(prompts, str):
        prompts = [prompts]

    h, w = image_pil.height, image_pil.width
    img_np = np.array(image_pil.convert("RGB"))
    img_hsv = cv2.cvtColor(img_np, cv2.COLOR_RGB2HSV)

    results_list = []

    for prompt in prompts:
        inputs = processor(images=image_pil, text=prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)

        res = processor.post_process_instance_segmentation(
            outputs, threshold=0.5, mask_threshold=0.5,
            target_sizes=[[h, w]]
        )[0]

        masks = res["masks"]
        scores = res.get("scores", [])
        if len(scores) == 0:
            scores = [1.0] * len(masks)

        # merge all masks
        merged = np.zeros((h, w), dtype=np.uint8)
        mask_data = []
        high_conf_masks = []
        for i in range(len(masks)):
            if scores[i] < 0.5:
                continue
            m = masks[i].cpu().numpy().astype(np.uint8)
            merged = cv2.bitwise_or(merged, m)
            high_conf_masks.append(m)
            mask_data.append({
                "prompt": prompt,
                "score": float(scores[i]),
                "area_px": int(np.sum(m > 0)),
            })

        # parameters
        total_pixels = h * w
        plant_pixels = np.sum(merged > 0)
        coverage_ratio = plant_pixels / total_pixels if total_pixels > 0 else 0

        rows = np.any(merged, axis=1)
        if rows.any():
            plant_top = int(np.argmax(rows))
            plant_bot = int(h - np.argmax(rows[::-1]))
            height_px = plant_bot - plant_top
        else:
            height_px = 0

        plant_region = img_hsv[merged > 0]
        mean_green = float(np.mean(img_np[merged > 0, 1])) if len(plant_region) > 0 else 0
        mean_hue = float(np.mean(plant_region[:, 0])) if len(plant_region) > 0 else 0

        leaf_count = len(high_conf_masks)

        # encode mask as base64 PNG
        _, mask_bytes = cv2.imencode(".png", merged * 255)
        mask_b64 = base64.b64encode(mask_bytes).decode("utf-8")

        results_list.append({
            "prompt": prompt,
            "leaf_count": leaf_count,
            "coverage_ratio": round(coverage_ratio, 4),
            "height_px": height_px,
            "mean_greenness": round(mean_green, 2),
            "mean_hue": round(mean_hue, 2),
            "total_plant_area_px": int(plant_pixels),
            "instances": mask_data,
            "mask_png_b64": mask_b64,
        })

    return results_list

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional
import uvicorn
import nest_asyncio

app = FastAPI(title="VitroVision SAM3 API")

class SegmentRequest(BaseModel):
    image: str  # base64 JPEG/PNG
    prompts: List[str]

class SegmentResponse(BaseModel):
    status: str
    results: list

@app.post("/segment")
async def segment(req: SegmentRequest):
    try:
        img_bytes = base64.b64decode(req.image)
        img = Image.open(BytesIO(img_bytes)).convert("RGB")
        results = segment_image(img, req.prompts)
        return SegmentResponse(status="ok", results=results)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/health")
async def health():
    return {"status": "ok"}

nest_asyncio.apply()
print("FastAPI server ready!")

In [ ]:
from pyngrok import ngrok

# เปิด public URL ผ่าน ngrok
public_url = ngrok.connect(8000).public_url
print(f"\n")
print(f"  API URL: {public_url}/segment")
print(f"  Health:  {public_url}/health")
print(f"\n")
print(f"ใส่ URL นี้ใน Android app config")

In [ ]:
uvicorn.run(app, host="0.0.0.0", port=8000)
# cell นี้จะรันต่อเนื่อง — อย่ากดหยุดถ้ายังใช้ API อยู่